# Information
## Source
This notebook prepares the data from Lawley et al. (2022) using the version from https://drive.google.com/file/d/1jyxbPmwhMEhgezxMTxwmKTuU1PhT9yPe using the original H3 hexagonal polygon data.

**Citaton**: <p>
Christopher J.M. Lawley, Anne E. McCafferty, Garth E. Graham, David L. Huston, Karen D. Kelley, Karol Czarnota, Suzanne Paradis, Jan M. Peter, Nathan Hayward, Mike Barlow, Poul Emsbo, Joshua Coyan, Carma A. San Juan, Michael G. Gadd: <br>
Data–driven prospectivity modelling of sediment–hosted Zn–Pb mineral systems and their critical raw materials. <br>
Ore Geology Reviews, Volume 141, 2022, 104635, ISSN 0169-1368, https://doi.org/10.1016/j.oregeorev.2021.104635.

## What is it for?
This notebook creates a base raster to be used for spatial alignment of other raster datasets. <br>
It is based on the H3 hexagonal polygons from Lawley et al. (2022) and is used to create a base raster for the study area.

In [11]:
# Standard libraries
import os
import numpy as np
from pathlib import Path
from importlib.resources import files
from rasterio.crs import CRS

# Custom modules
from beak.utilities import io, conversions
from beak.utilities.conversions import _rasterize_vector_process, transform_from_geometries
from beak.utilities.io import save_raster, check_path
from beak.utilities.misc import replace_invalid_characters


**User inputs**

In [12]:
# Path to datacube
BASE_PATH = files("beak.data")
PATH_DATACUBE = BASE_PATH / "LAWLEY22" / "RAW" / "2021_Table04_Datacube.csv"
FEATHER = PATH_DATACUBE.parent / str(PATH_DATACUBE.stem + ".feather")

# ROI
REGIONS = ["United States of America", "Canada"]    # Canada, United States of America, Australia
N_ROWS = None                                       # Number of rows to read from datacube, None for all
FILTER_LONG = None                                  # Maximum longitude to consider
FILTER_LAT = None                                   # Maximum latitude to consider

# Coordinate system
EPSG_SRC = 4326

# Resolution according to the CRS
RES_SRC = 0.025                                     # Degreees: 0.015° ~ 2 sqkm, 0.02° ~ 3.75 sqkm, 0.025° ~ 6 sqkm

# COLUMN NAMES
COL_LONGITUDE = "Longitude_EPSG4326"                # Longitude column in datacube
COL_LATITUDE = "Latitude_EPSG4326"                  # Latitude column in datacube
COL_POLYGONS = "H3_Geometry"                        # Geometry column in datacube

# Create string with spatial attributes for subfolder naming
SPATIAL_EXTENT = "US_CANADA"
SPATIAL_DEFINITION = str("EPSG_" + str(EPSG_SRC) + "_RES_" +  replace_invalid_characters(str(RES_SRC)))

# Path for base raster to be created
OUT_FILE = BASE_PATH / "BASE_RASTERS" / str(SPATIAL_DEFINITION + "_" + SPATIAL_EXTENT + ".tif")

# Overwrite the path for testing purposes
CURRENT_DIR = Path(os.getcwd())
OUT_FILE = CURRENT_DIR / "out_file.tif"

# Check save path for base raster
print(f"Datacube file (input): {PATH_DATACUBE}")     
print(f"Base raster file (export and import): {OUT_FILE}")
                               

Datacube file (input): s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\LAWLEY22\RAW\2021_Table04_Datacube.csv
Base raster file (export and import): s:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\local\experiments\data_preprocessing\preparation_base_rasters\us_canada.tif


Load RAW datacube with selection to U.S. and Canada

In [3]:
# Load
if FEATHER.exists() == True:
  print("Loading feather...")
  df = io.load_feather(FEATHER, nrows=N_ROWS)
else:
  print("Loading dataset...")
  df = io.load_dataset(PATH_DATACUBE, nrows=N_ROWS)

# Filter ROI
df = df[df["Country_Majority"].isin(REGIONS)]

# Replace "-"
df.replace("-", np.nan, inplace=True)

# Shape
print(df.shape)
  

Loading feather...
(3620129, 97)


# Export to raster format

**Spatial filtering**

In [6]:
# Filter data with latitudes in the range of the target region
df_sf = io.spatial_filter(df, longitude_column=COL_LONGITUDE, longitude_max=FILTER_LONG)


**Convert H3 polygons and create geodataframe**

In [7]:
# Create geodataframe in advance
gdf = conversions.create_geodataframe_from_polygons(data=df_sf, polygon_col=COL_POLYGONS, epsg_code=EPSG_SRC)


## Export **base raster**

**Rasterize dataframe**

In [8]:
# Rasterize a column with all ones
print("Rasterize coverage column...")
width, height, transform = transform_from_geometries(gdf, RES_SRC)

cov_column, cov_raster, cov_transform = _rasterize_vector_process(
    value_column="COVERAGE",
    values=np.ones(len(gdf), dtype=np.int8),
    geometries=gdf.geometry.values,
    height=height,
    width=width, 
    nodata_value=-99,
    transform=transform,
    all_touched=False,
    merge_strategy="replace",
    default_value=0,
    dtype=np.int8,
    impute_nodata=False,
)

Rasterize coverage column...


**Save raster**

In [13]:
save_folder = check_path(OUT_FILE.parent)
print("Save to: ", save_folder)

save_raster(
            path=OUT_FILE,
            array=cov_raster,
            crs=CRS.from_epsg(EPSG_SRC),
            height=height,
            width=width,
            nodata_value=-99,
            transform=cov_transform,
)
      

Save to:  s:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\local\experiments\data_preprocessing\preparation_base_rasters
